In [32]:
import os
import json
import dspy
from datasets import DatasetDict, Dataset, concatenate_datasets
from create_datasets import convert_tokens_to_entities
import numpy as np

cwd = os.getcwd()

In [3]:
model="Qwen/Qwen3.5-9B"
# vllm serve Qwen/Qwen3.5-9B --port 8000 --tensor-parallel-size 1 --max-model-len 262144 --reasoning-parser qwen3 
# recommended: --max-model-len >= 128000
#8000 for gpu0
#8005 for gpu1
lm = dspy.LM(f"openai/{model}",
             api_base="http://localhost:8005/v1",  # ensure this points to your port
             api_key="local", model_type="chat", max_tokens=10000)
dspy.configure(lm=lm)

In [ ]:
class PolProc(dspy.Signature):
    """Extract entities in the provided policy text for each policy design element category: 'Actor', 'InstrumentType', 'Objective', 'Resource', and 'Time'. If there are no entities for a class, return an empty list. Include reasoning and step-by-step thinking before producing final output."""
    text: str = dspy.InputField(desc="Text from a policy document.")
    reasoning: str = dspy.OutputField(
        desc="Step-by-step reasoning and thinking."
    )
    output: str = dspy.OutputField(desc="Dictionary of the list of entities found for each class.")

In [5]:
test_proc = dspy.Predict(PolProc)

In [11]:
pred = test_proc(text="The commission will allocate 5% of the national budget for investment in LESS equipment.")
print(pred)
#mml128000: 4m14s
#mml262144: 2m20s

Prediction(
    reasoning='1.  **Analyze the Actor**: The sentence begins with "The commission will...", identifying "The commission" as the subject performing the action. This maps to the \'Actor\' category.\n2.  **Analyze the Instrument**: The phrase "allocate 5% of the national budget" describes the mechanism or method used to implement the policy. This is a fiscal allocation, which serves as the \'InstrumentType\'.\n3.  **Analyze the Objective**: The text mentions "for investment in LESS equipment". While this describes the use of funds, it specifies the activity rather than an explicit outcome or goal (e.g., "to reduce emissions" or "to improve safety"). Therefore, no specific \'Objective\' can be extracted from this sentence.\n4.  **Analyze the Resource**: The specific financial resource mentioned is "5% of the national budget". This maps to the \'Resource\' category.\n5.  **Analyze the Time**: The sentence uses the future tense "will allocate", but there are no specific temporal

In [13]:
print(pred.output)
#mml128000 = {"Actor": ["We, the undersigned"], "InstrumentType": ["Alpine Climate Target System 2050", "Climate Action Plan 2.0"], "Objective": ["combat climate change", "climate-neutral and climate-resilient Alps"], "Resource": [], "Time": ["2050"]}
#mml262144 = {"Actor": ["We, the undersigned"], "InstrumentType": ["Alpine Climate Target System 2050", "Climate Action Plan 2.0"], "Objective": ["combat climate change", "climate-neutral and climate-resilient Alps"], "Resource": [], "Time": ["2050"]}

{"Actor": ["The commission"], "InstrumentType": ["allocate 5% of the national budget"], "Objective": [], "Resource": ["5% of the national budget"], "Time": []}


In [14]:
r=0
dsdct_dir = f"{cwd}/inputs/a/mhead_dsdcts"
dataset_dict = DatasetDict.load_from_disk(f"{dsdct_dir}/dsdct_r{r}")

In [15]:
dataset_dict['test'][0]['text']

'article\xa021\r\nspecific provisions related to energy from renewable sources in transport\r\n1.\xa0\xa0\xa0member states shall ensure that information is given to the public on the availability and environmental benefits of all different renewable sources of energy for transport. when the percentages of biofuels, blended in mineral oil derivatives, exceed 10\xa0% by volume, member states shall require this to be indicated at the sales points.\r\n2.\xa0\xa0\xa0for the purposes of demonstrating compliance with national renewable energy obligations placed on operators and the target for the use of energy from renewable sources in all forms of transport referred to in article\xa03(4), the contribution made by biofuels produced from wastes, residues, non-food cellulosic material, and ligno-cellulosic material shall be considered to be twice that made by other biofuels.\r\n'

Restructure Datasets

In [17]:
ds = dataset_dict['test']
ds

Dataset({
    features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
    num_rows: 82
})

In [18]:
en_ds = convert_tokens_to_entities(ds)
en_ds

Dataset({
    features: ['id', 'text', 'tokens', 'Actor', 'InstrumentType', 'Objective', 'Resource', 'Time'],
    num_rows: 82
})

In [35]:
def entity_ds_to_dspy_ds(en_ds):
    examples = []
    for entry in en_ds:
        example = dspy.Example(
            text = entry['text'],
            output = {lbl: entry[lbl] for lbl in list(entry) if lbl not in ["id","text","tokens"]}
        ).with_inputs('text')
        examples.append(example)
    return examples
en_ds = concatenate_datasets([dataset_dict['train'], dataset_dict['dev']])

In [ ]:
def dspy_icl_eval(example, pred, catch):
    try:
        f1s = []
        for label in list(example):
            gold_set = example[label]
            pred_set = pred[label]
            res = { "tp": 0, "fp": 0, "fn": 0, "n_gold": 0, "n_pred": 0}
            res['tp'] = len(gold_set.intersection(pred_set))
            res['fn'] = len(gold_set - pred_set)
            res['fp'] = len(pred_set - gold_set)
            res['n_gold'] = len(gold_set)
            res['n_pred'] = len(pred_set)
            pres = res['tp'] / res['n_pred'] if res['n_pred'] > 0 else 0
            reca = res['tp'] / res['n_gold'] if res['n_gold'] > 0 else 0
            if pres + reca > 0:
                f1 = 2 * pres * reca / (pres + reca)
            else:
                f1 = 0  
            scores =  {
                'precision': pres,
                'recall': reca,
                'f1': f1
            }
            f1s.append(scores['f1'])
        return np.average(f1s)
    except Exception as exp:
        print(exp)
        return 0.0

In [37]:
from dspy.teleprompt import MIPROv2
model_name = "Qwen/Qwen3.5-9B"
model_name = model.split('/')[-1]
mode = "a"
interest = "zero-shot"
lvl = "art"
n_icl = 5
results_addr = f"{cwd}/results/{mode}/dspy/{interest}/{lvl}"
model_addr = f"{cwd}/models/{mode}/dspy/{interest}/{lvl}"
os.makedirs(model_addr, exist_ok=True)
test_proc = dspy.Predict(PolProc)
teleprompter = MIPROv2(
            metric=dspy_icl_eval,
            auto='light',#"medium",
            max_bootstrapped_demos=5, 
            max_labeled_demos=n_icl)
dds = entity_ds_to_dspy_ds(en_ds)
optimized_program = teleprompter.compile(
            test_proc,
            trainset=dds)
model_save_path = os.path.join(model_addr, f"{model_name}_{n_icl}.json")
optimized_program.save(model_save_path)

2026/04/05 15:19:16 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/04/05 15:19:16 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 100

2026/04/05 15:19:16 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/04/05 15:19:16 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/04/05 15:19:16 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


  0%|          | 0/66 [03:03<?, ?it/s]


KeyboardInterrupt: 